In [0]:
%sql
-- task 1
CREATE VOLUME IF NOT EXISTS workspace.coffee.landing;

In [0]:
# task 1
LANDING = "/Volumes/workspace/coffee/landing/online_orders"
CHECKPOINT = "/Volumes/workspace/coffee/landing/_checkpoints/bronze_online_orders"

dbutils.fs.mkdirs(LANDING)
display(dbutils.fs.ls("/Volumes/workspace/coffee/landing"))

In [0]:
# task 2
import json

def land(file_name, orders):
    text = "\n".join(json.dumps(o) for o in orders)
    dbutils.fs.put(f"{LANDING}/{file_name}", text, overwrite=True)

In [0]:
batch_001 = [
    {"order_id": 201, "customer_id": 10, "city": "Berlin", "order_ts": "2026-02-01T09:15:00",
     "items": [{"product": "grinder", "qty": 1, "price_eur": 119.0},
               {"product": "filter", "qty": 2, "price_eur": 12.5}]},
    {"order_id": 202, "customer_id": 11, "city": " hamburg ", "order_ts": "2026-02-01T10:02:00",
     "items": [{"product": "mug", "qty": 2, "price_eur": 8.9}]},
    {"order_id": 203, "customer_id": 12, "city": "Berlin", "order_ts": "2026-02-01T11:30:00",
     "items": [{"product": "kettle", "qty": 1, "price_eur": 45.0}]},
]

land("batch_001.json", batch_001)

In [0]:
# task 3
from pyspark.sql.functions import col, current_timestamp

bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(LANDING)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp()))

query = (bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.coffee.bronze_online_orders"))

query.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) FROM workspace.coffee.bronze_online_orders;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.coffee.bronze_online_orders;

SELECT * FROM workspace.coffee.bronze_online_orders;

DESCRIBE TABLE workspace.coffee.bronze_online_orders;

In [0]:
# task 4
display(dbutils.fs.ls(CHECKPOINT))

In [0]:
# task 5
batch_002 = [
    {"order_id": 203, "customer_id": 12, "city": "Berlin", "order_ts": "2026-02-01T11:30:00",
     "items": [{"product": "kettle", "qty": 1, "price_eur": 45.0}]},
    {"order_id": 204, "customer_id": 13, "city": "Hamburg", "order_ts": "2026-02-02T14:20:00",
     "items": [{"product": "scale", "qty": 1, "price_eur": 39.0}]},
    {"order_id": 205, "customer_id": 10, "city": "BERLIN", "order_ts": "2026-02-02T16:05:00",
     "items": [{"product": "filter", "qty": 3, "price_eur": 12.5}]},
]

land("batch_002.json", batch_002)

In [0]:
# task 3
from pyspark.sql.functions import col, current_timestamp

bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(LANDING)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp()))

query = (bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.coffee.bronze_online_orders"))

query.awaitTermination()

In [0]:
%sql
SELECT source_file, COUNT(*) AS orders
FROM workspace.coffee.bronze_online_orders
GROUP BY source_file
ORDER BY source_file;

In [0]:
%sql
-- task 6
SELECT order_id, COUNT(*) AS copies
FROM workspace.coffee.bronze_online_orders
GROUP BY order_id
ORDER BY order_id;



In [0]:
batch_003 = [
    {"order_id": 206, "customer_id": 14, "city": "Berlin", "order_ts": "2026-02-03T08:45:00",
     "channel": "app",
     "items": [{"product": "grinder", "qty": 1, "price_eur": 119.0},
               {"product": "mug", "qty": 1, "price_eur": 8.9}]},
    {"order_id": 207, "customer_id": "unknown", "city": "Hamburg", "order_ts": "2026-02-03T09:10:00",
     "channel": "app",
     "items": [{"product": "filter", "qty": 4, "price_eur": 12.5}]},
]

land("batch_003.json", batch_003)

In [0]:
from pyspark.sql.functions import col, current_timestamp

bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(LANDING)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp()))

query = (bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.coffee.bronze_online_orders"))

query.awaitTermination()

In [0]:
from pyspark.sql.functions import col, current_timestamp

bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(LANDING)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp()))

query = (bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.coffee.bronze_online_orders"))

query.awaitTermination()

In [0]:
%sql
SELECT order_id, customer_id, channel, _rescued_data
FROM workspace.coffee.bronze_online_orders
ORDER BY order_id;

In [0]:
%sql
SELECT order_id, customer_id, _rescued_data
FROM workspace.coffee.bronze_online_orders
WHERE _rescued_data IS NOT NULL;

In [0]:
%sql
-- task 12
CREATE TABLE IF NOT EXISTS workspace.coffee.bronze_copy_into;

COPY INTO workspace.coffee.bronze_copy_into
FROM '/Volumes/workspace/coffee/landing/online_orders'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

SELECT COUNT(*) FROM workspace.coffee.bronze_copy_into;

DESCRIBE TABLE workspace.coffee.bronze_copy_into;

In [0]:
%sql
-- task 13
COPY INTO workspace.coffee.bronze_copy_into
FROM '/Volumes/workspace/coffee/landing/online_orders'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

SELECT COUNT(*) FROM workspace.coffee.bronze_copy_into;

DROP TABLE workspace.coffee.bronze_copy_into;